In [ ]:
"""
    Training data
        ↓
    Llama 3.1 8B
        ↓
    4-bit quantization
        ↓
    Add LoRA adapters
        ↓
    Train adapters
        ↓
    Save adapter
        ↓
    Push adapter to Hugging Face
"""
# Why do we need tokenizer - Llama cannot directly understand Python strings. It works with numbers (token IDs). 
# The tokenizer converts your text into those numbers. PyTorch provides the numerical/AI computation framework, and 
# the other imports provide the dataset and model-loading tools.

import torch                          # << ---Does the AI/math computation
from datasets import load_dataset     # <<---Loads training data
from transformers import (
    AutoTokenizer,                    # <<----Converts text ↔ token numbers
    AutoModelForCausalLM,             # <<-------Loads the Llama model
    BitsAndBytesConfig                # <<--Configures 4-bit quantization
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig



model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)     #<<--This downloads/loads the tokenizer that belongs to your Llama model.
tokenizer.pad_token = tokenizer.eos_token                                # <<-- EoS - End of sequence
"""
Sometimes we have multiple pieces of text with different lengths:
"I like AI"
"I like AI because it is useful"

The model may need them to have the same length, so shorter text is padded:
"I like AI" + [PAD] [PAD] [PAD]
"""
# Load Llama 3.1 in a memory-saving 4-bit format and automatically put it on the available GPU/CPU.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # << --Store model weights using 4 bits instead of higher precision
    bnb_4bit_quant_type="nf4",              # << --Use NF4 as the 4-bit format
    bnb_4bit_compute_dtype=torch.bfloat16   # << - Do calculations using bfloat16
)

model = AutoModelForCausalLM.from_pretrained(  #<<- "Download/load the model specified by model_name, using the 4-bit settings I just created.
    model_name,
    quantization_config=quant_config,
    device_map="auto"                          # << - Automatically decide where to put the model (GPU or CPU)
)


# --------loRA Configuration------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)

dataset = load_dataset(
    "json",
    data_files="train.json"
)

#----------Training configuration parameters
training_args = SFTConfig(
    output_dir="./llama-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True
)
#-----------------Below is the trainer----------------
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
    peft_config=lora_config
)
trainer.train()
trainer.save_model("./llama-lora")